# 07 — DRL v4: PPO Entry-Timer with Mechanical ATR-Bracket Exits

**The reframe.** v0–v3 lost money *even at zero fees* (−17% OOS) because the training
env let the policy pick a new position every bar → a per-bar churner on an H≈0.52
series, which the bracket backtest then re-entered 2 484 times.

v4 decouples **direction** from **holding**, exactly like the LGBM agent that works:

| | v3 (MarketEnv) | v4 (BracketMarketEnv) |
|---|---|---|
| Agent decides | position every bar | **only WHEN to enter** |
| Exit | agent flips / penalty | **mechanical** TP/SL/max-hold, trade *locked* |
| Reward | per-step PnL (noise) | **bracketed-trade net return** (fees+funding in) |
| Churn | structural (2 484 trades) | **impossible** — locked until bracket resolves |

The environment is the single source of truth: the same brackets are used in training
and evaluation (`agent.step_returns_` / `agent.trades_` come straight from the policy
rollout), so there is no second, divergent backtest layer.

> **Compute note.** A full run is ~30 folds × `TOTAL_TIMESTEPS` PPO steps. Start with
> `TOTAL_TIMESTEPS=50_000` for a smoke pass (~minutes); raise to `500_000` for the
> real result. This notebook ships **unexecuted**.

In [ ]:
import json, time, calendar, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from hmats.agents.drl_bracket_agent import DRLBracketAgent, _DEFAULT_BRACKET_FEATURES
from hmats.agents.meta_agent import run_sized_backtest
from hmats.viz.plots import save_fig

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
try:    plt.style.use('seaborn-v0_8-whitegrid')
except: plt.style.use('seaborn-whitegrid')
mpl.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],
    'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':120,
    'savefig.dpi':150,'savefig.bbox':'tight'})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'; RED='#EF5350'; GREEN='#26A69A'; PURPLE='#7B1FA2'

# ── WFO / OOS config (mirrors LGBM v12 / DRL v3) ──────────────────────────────
OOS_START       = pd.Timestamp('2024-01-01')
TRAIN_WINDOW_H  = 8760
STEP_SIZE       = 720
TOTAL_TIMESTEPS = 500_000     # smoke pass: set 50_000

# ── Bracket mechanics (identical in training and eval) ────────────────────────
WINDOW_SIZE = 24
EPISODE_LEN = 1000
SL_MULT, TP_MULT = 2.0, 3.0
MIN_HOLD, MAX_HOLD, COOLDOWN = 4, 24, 2
ENT_COEF = 0.02

TAKER_FEE = 0.0005; FUNDING_H = 0.0000077

print('DRL v4 — entry-timer with mechanical brackets')
print(f'  brackets: SL={SL_MULT}xATR TP={TP_MULT}xATR  min_hold={MIN_HOLD} max_hold={MAX_HOLD} cooldown={COOLDOWN}')
print(f'  PPO ts/fold={TOTAL_TIMESTEPS:,}  ent_coef={ENT_COEF}')
print('Imports OK')

In [ ]:
def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise RuntimeError('repo root not found')

REPO_DIR = _find_repo_root()
FEAT_DIR = REPO_DIR / 'data' / 'features'
EXT_DIR  = REPO_DIR / 'data' / 'external'
ARTS_DIR = REPO_DIR / 'artifacts' / '07_drl_omni_0fee_v4'
ARTS_DIR.mkdir(parents=True, exist_ok=True)
print('Artifacts →', ARTS_DIR)

In [ ]:
# ── Load features (V1 + V4 + raw H/L + microstructure) ───────────────────────
print('Loading V1...');  v1 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
v1.index = v1.index.tz_localize(None) if v1.index.tz else v1.index
print('Loading V4...');  v4 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet')
v4.index = v4.index.tz_localize(None) if v4.index.tz else v4.index

merged = v1.copy()
_raw = pd.read_parquet(REPO_DIR / 'data' / 'raw' / 'BTCUSDT_1h.parquet')
_raw.index = _raw.index.tz_convert(None)
merged['high'] = _raw['high'].reindex(merged.index)
merged['low']  = _raw['low'].reindex(merged.index)
for col in ['close_vs_true_vwap','hurst_24h','hurst_72h','tfi_pct','tfi_z_24h','bb_width_pct']:
    if col in v4.columns: merged[col] = v4[col].reindex(merged.index)

# Phase-K microstructure features (00_data_ingestion_v4)
_mp = FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet'
if _mp.exists():
    micro = pd.read_parquet(_mp)
    micro.index = micro.index.tz_localize(None) if micro.index.tz else micro.index
    for col in micro.columns: merged[col] = micro[col].reindex(merged.index)
    print('Microstructure features merged:', list(micro.columns))
else:
    print('WARNING: microstructure parquet missing — run 00_data_ingestion_v4 Phase K')

present = [f for f in _DEFAULT_BRACKET_FEATURES if f in merged.columns]
missing = [f for f in _DEFAULT_BRACKET_FEATURES if f not in merged.columns]
print(f'Features present: {len(present)}/{len(_DEFAULT_BRACKET_FEATURES)}', '| missing:', missing)
oos_df = merged[merged.index >= OOS_START].copy()
print(f'Total bars: {len(merged):,}  |  OOS bars: {len(oos_df):,}')

In [ ]:
# ── PHASE 1 — M1Y WFO: entry-timer policy ────────────────────────────────────
print('='*60); print('PHASE 1 — DRL v4 entry-timer WFO'); print('='*60)

agent = DRLBracketAgent(
    features=[f for f in _DEFAULT_BRACKET_FEATURES if f in merged.columns],
    window_size=WINDOW_SIZE, episode_len=EPISODE_LEN,
    sl_mult=SL_MULT, tp_mult=TP_MULT,
    min_hold=MIN_HOLD, max_hold=MAX_HOLD, cooldown=COOLDOWN,
    train_window_h=TRAIN_WINDOW_H, step_size=STEP_SIZE,
    total_timesteps=TOTAL_TIMESTEPS, agent_id='drl_bracket_v4',
    ppo_kwargs={'ent_coef': ENT_COEF},
)
t0 = time.time()
signals = agent.generate_signals(merged, oos_start=OOS_START, verbose=True)
print(f'Done in {(time.time()-t0)/60:.1f} min')

signals.to_frame().to_parquet(ARTS_DIR / 'drl_oos_signals.parquet')
agent.step_returns_.to_frame().to_parquet(ARTS_DIR / 'drl_step_returns.parquet')
_lab = {-1:'Short', 0:'Flat', 1:'Long'}
for k, v in signals.value_counts().sort_index().items():
    print(f'  {_lab.get(int(k), k):5s}: {v:6,}  ({v/len(signals)*100:.1f}%)')
_oos_entries = sum(1 for t in agent.trades_ if merged.index[t['entry_bar']] >= OOS_START)
print(f'OOS trades (entries): {_oos_entries}')

In [ ]:
# ── PHASE 2 — Backtest (env is source of truth) ──────────────────────────────
print('='*60); print('PHASE 2 — OOS BACKTEST'); print('='*60)

def _sharpe(eq):
    r = np.diff(np.log(np.maximum(eq,1e-12)));  return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(24*365))
def _maxdd(eq):
    pk = np.maximum.accumulate(eq);  return float(((eq-pk)/(pk+1e-12)).min())

# signals is OOS-only (full_history=False); align everything to its index
oos_index = signals.index
pos_oos = signals.astype(float)

# WITH FEES: straight from the policy rollout's per-bar net returns (fees+funding baked in)
sret_oos = agent.step_returns_.loc[oos_index].values
eq_fees = np.cumprod(1.0 + sret_oos)

# ZERO-FEE upper bound: replay the position path with taker_fee=0 (close-to-close)
ohlcv_oos = merged.loc[oos_index, ['close','high','low','atr_14_pct']].copy()
eq_0fee, _ = run_sized_backtest(pos_oos, ohlcv_oos, taker_fee=0.0, funding_h=FUNDING_H)

# Trade-level stats from the env trade log (OOS only)
tr = [t for t in agent.trades_ if merged.index[t['entry_bar']] >= OOS_START]
trdf = pd.DataFrame(tr) if tr else pd.DataFrame(columns=['direction','ret','hold','entry_bar','exit_bar'])
n_l = int((trdf['direction']=='long').sum())  if len(trdf) else 0
n_s = int((trdf['direction']=='short').sum()) if len(trdf) else 0
wr  = float((trdf['ret']>0).mean()) if len(trdf) else 0.0
avg_hold = float(trdf['hold'].mean()) if len(trdf) else 0.0

print(f"{'':20}  {'Trades':>7} {'L/S':>9} {'Win':>6} {'Return':>8} {'Sharpe':>7} {'MaxDD':>7}")
print('-'*70)
print(f"{'With fees (primary)':20}  {len(trdf):>7} {n_l:>4}/{n_s:<4} {wr:>6.1%} {eq_fees[-1]-1:>+7.1%} {_sharpe(eq_fees):>7.3f} {_maxdd(eq_fees):>7.2%}")
print(f"{'Zero-fee (upper)':20}  {len(trdf):>7} {n_l:>4}/{n_s:<4} {wr:>6.1%} {eq_0fee[-1]-1:>+7.1%} {_sharpe(eq_0fee):>7.3f} {_maxdd(eq_0fee):>7.2%}")
print(f'\navg hold = {avg_hold:.1f} bars  | trades/month ~ {len(trdf)/max((oos_index[-1]-oos_index[0]).days/30,1):.1f}')
print('vs DRL v3: 2484 trades, -93% w/fees, -17% zero-fee')

In [ ]:
# ── PHASE 3 — Equity vs Buy&Hold ─────────────────────────────────────────────
bh = (ohlcv_oos['close'].values / ohlcv_oos['close'].values[0] - 1) * 100
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(oos_index, (eq_fees-1)*100, color=PURPLE, lw=1.6, label=f'DRL v4 w/fees  Sharpe={_sharpe(eq_fees):.2f}  ret={eq_fees[-1]-1:+.1%}')
ax.plot(oos_index, (eq_0fee-1)*100, color=BLUE, lw=1.0, ls='--', alpha=0.8, label=f'DRL v4 0-fee  ret={eq_0fee[-1]-1:+.1%}')
ax.plot(oos_index, bh, color=GREY, lw=1.0, ls=':', alpha=0.7, label='BTC B&H')
ax.axhline(0, color=GREY, lw=0.5, ls=':')
ax.set_ylabel('Return (%)'); ax.legend(fontsize=8, loc='upper left')
ax.set_title('DRL v4 entry-timer — OOS equity', fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.tight_layout(); save_fig(fig, ARTS_DIR / '01_equity.png'); plt.show()

In [ ]:
# ── Action distribution + hold histogram ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13,4))
ac = pd.Series(signals).value_counts().sort_index()
lab = {-1:'Short',0:'Flat',1:'Long'}; col = {-1:RED,0:GREY,1:GREEN}
axes[0].bar([lab[k] for k in ac.index], ac.values, color=[col[k] for k in ac.index], alpha=0.8)
axes[0].set_title('OOS position-bar distribution', fontweight='bold')
if len(trdf):
    axes[1].hist(trdf['hold'], bins=24, color=PURPLE, alpha=0.8)
    axes[1].axvline(MAX_HOLD, color=RED, ls='--', lw=1, label=f'max_hold={MAX_HOLD}')
    axes[1].axvline(MIN_HOLD, color=GREEN, ls='--', lw=1, label=f'min_hold={MIN_HOLD}')
    axes[1].set_title('Trade hold duration (bars)', fontweight='bold'); axes[1].legend(fontsize=8)
fig.tight_layout(); save_fig(fig, ARTS_DIR / '02_actions_holds.png'); plt.show()

In [ ]:
# ── Save results.json ────────────────────────────────────────────────────────
def _m(eq, label):
    return {'label':label,'n_trades':int(len(trdf)),'n_long':n_l,'n_short':n_s,
            'win_rate':round(wr,4),'avg_hold_h':round(avg_hold,2),
            'total_ret':round(float(eq[-1]-1),4),'sharpe':round(_sharpe(eq),4),'maxdd':round(_maxdd(eq),4)}

results = {
    'notebook':'07_drl_omni_0fee_v4','version':'v4',
    'created':pd.Timestamp.now().isoformat(),
    'algorithm':'PPO entry-timer + mechanical ATR brackets (BracketMarketEnv)',
    'reframe':'agent decides WHEN to enter; exit is mechanical TP/SL/max-hold; trade locked (no churn)',
    'brackets':{'sl_mult':SL_MULT,'tp_mult':TP_MULT,'min_hold':MIN_HOLD,'max_hold':MAX_HOLD,'cooldown':COOLDOWN},
    'ppo':{'total_timesteps':TOTAL_TIMESTEPS,'ent_coef':ENT_COEF,'window_size':WINDOW_SIZE,'episode_len':EPISODE_LEN},
    'oos_period':f'{OOS_START.date()} → {oos_index[-1].date()}',
    'n_features':len(agent.features),
    'fees':{'taker':TAKER_FEE,'funding_h':FUNDING_H},
    'backtest_wfees':_m(eq_fees,'w_fees'),
    'backtest_0fee': _m(eq_0fee,'0_fee'),
    'baseline_v3':{'n_trades':2484,'total_ret':-0.9296,'sharpe':-3.1209,'zero_fee_ret':-0.1718},
}
with open(ARTS_DIR / 'results.json','w') as f: json.dump(results, f, indent=2)
print('Saved →', ARTS_DIR / 'results.json'); print(json.dumps(results, indent=2))